# Module 6 Lab 3: Appendix C.3

### Searching and Sorting My Customers Table

**Name:** Mac Mckay

**Course:** CS 82A

**Repo:** https://github.com/raulmckay-blip/cs82a-portfolio

## Overview

This project takes my Lab 2 customers table and filters, sorts, and searches it with SQL, then charts one result.

## Rebuild the table from Lab 2

Same code as Lab 2 so this runs on its own. `':memory:'` keeps it in memory, no file saved.

In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

con = sqlite3.connect(':memory:')

con.execute('''
    CREATE TABLE customers (
        id          INTEGER PRIMARY KEY,
        name        TEXT    NOT NULL DEFAULT '',
        city        TEXT    NOT NULL DEFAULT '',
        signup_year INTEGER NOT NULL DEFAULT 0,
        total_spent REAL    NOT NULL DEFAULT 0
    )
''')

con.execute('''
    INSERT INTO customers (name, city, signup_year, total_spent)
    VALUES
        ('Ousmane Dembele',   'Culver City',  2021, 412.50),
        ('Rodri',             'Santa Monica', 2023, 88.25),
        ('Lionel Messi',      'Venice',       2022, 1290.00),
        ('Karim Benzema',     'Inglewood',    2024, 57.80),
        ('Luka Modric',       'Los Angeles',  2020, 760.40),
        ('Cristiano Ronaldo', 'Torrance',     2025, 15.99),
        ('Kaka',              'Culver City',  2022, 534.10),
        ('Fabio Cannavaro',   'Los Angeles',  2026, 199.95),
        ('Ronaldinho',        'Santa Monica', 2023, 0.00),
        ('Andriy Shevchenko', 'Venice',       2021, 342.75)
''')
con.commit()

pd.read_sql("SELECT COUNT(*) AS n_rows FROM customers", con)

10 rows, same as Lab 2.

## Query 1: Filter with WHERE

Like the book's `Region = 'Caribbean'`, but for Venice:

In [ ]:
pd.read_sql('''
    SELECT name, city, total_spent
    FROM customers
    WHERE city = 'Venice'
''', con)

Lionel Messi and Andriy Shevchenko. Text has to match exactly, so `'venice'` would return nothing.

## Query 2: Sort with ORDER BY

Customers who signed up in 2023 or later, lowest spender first:

In [ ]:
pd.read_sql('''
    SELECT name, signup_year, total_spent
    FROM customers
    WHERE signup_year >= 2023
    ORDER BY total_spent
''', con)

`DESC` flips it:

In [ ]:
pd.read_sql('''
    SELECT name, signup_year, total_spent
    FROM customers
    WHERE signup_year >= 2023
    ORDER BY total_spent DESC
''', con)

Cannavaro spent the most. Ronaldinho hasn't spent anything yet.

## Query 3: Text search with LIKE

`%` means any letters (or none), so `'%ar%'` finds "ar" anywhere:

In [ ]:
pd.read_sql('''
    SELECT name, city
    FROM customers
    WHERE name LIKE '%ar%'
''', con)

Karim Benzema and Fabio Cannavaro.

`%` only at the front means the name has to end with it:

In [ ]:
pd.read_sql('''
    SELECT name, city
    FROM customers
    WHERE name LIKE '%o'
''', con)

Ronaldo, Cannavaro, Ronaldinho, and Shevchenko all end in o. And `'r%'` means starts with r:

In [ ]:
pd.read_sql('''
    SELECT name, city
    FROM customers
    WHERE name LIKE 'r%'
''', con)

Found Rodri and Ronaldinho even with a lowercase r. `LIKE` ignores case, `=` doesn't.

## Query to chart: total spent by city

Same round trip as Lab 1: query, DataFrame, chart.

In [ ]:
city_totals = pd.read_sql('''
    SELECT city,
           COUNT(*)         AS n_customers,
           SUM(total_spent) AS total_spent
    FROM customers
    GROUP BY city
    ORDER BY total_spent DESC
''', con)

city_totals

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=city_totals, x='city', y='total_spent', color='steelblue')
ax.bar_label(ax.containers[0], fmt='${:,.0f}')
plt.title('Venice customers have spent the most, Torrance the least')
plt.xlabel('Customer city')
plt.ylabel('Total spent ($)')
plt.show()

Venice is on top, mostly thanks to Lionel Messi (\$1,290). With only 1 or 2 customers per city, one big spender changes everything.

In [ ]:
con.close()

## AI Use

I used AI to help with syntax and refine wording on this lab.